# Strategy backtester
Edit the **Config** cell, then run all. Works for any strategy fed by 1s `DydxSecondSnapshot`. Orders fill at the snapshot's best bid/ask (see `strategies/snapshot_backtest.py`).

In [ ]:
import sys
from decimal import Decimal
from pathlib import Path

import pandas as pd

TROLL = Path.cwd().resolve()
while TROLL.name != "platform":
    TROLL = TROLL.parent
sys.path.insert(0, str(TROLL))  # strategies are imported by "ml_signals.…" string path

from ml_signals.strategies.snapshot_backtest import run

CATALOG = str(TROLL / "dydx_collector" / "catalog")

In [ ]:
# ---- Config: change these and re-run the cells below ----
SYMBOL = "BTC-USD-PERP.DYDX"
START, END = "2026-09-05", "2026-09-06"   # bounded (MEM-01); the catalog has gaps, see note below

STRATEGY = "ml_signals.strategies.ofi_strategy:OFIStrategy"
STRATEGY_CONFIG = "ml_signals.strategies.ofi_strategy:OFIStrategyConfig"
PARAMS = {"trade_size": Decimal("0.01"), "ofi_threshold": 2.0, "warmup_seconds": 600}
# instrument_id is filled in automatically

In [ ]:
result = run(CATALOG, SYMBOL, START, END, STRATEGY, STRATEGY_CONFIG, PARAMS)
print(f"Events: {result.iterations}")
pd.DataFrame({"PnL": result.stats_pnls["USD"]}).join(
    pd.DataFrame({"Returns": result.stats_returns}), how="outer")

**Catalog coverage is patchy** (collector wasn't running continuously). Densest BTC days: 2026-09-05, 09-04, 09-19, 06-30, 07-02/03. A day with little data gives few events and no trades.